# LabelInspect - GPU access check

This checks the runtime, not the paper model. No private data or tokens are required.

**Kaggle:** import this notebook, select an available GPU in Settings > Accelerator, then run the cells. Complete account verification directly with Kaggle if required. **Colab fallback:** upload this notebook and choose a GPU runtime. Availability is controlled by the provider.

Only keep the GPU enabled while doing GPU work. Save `gpu_environment.json` after running.

In [ ]:
import json, platform, sys
from pathlib import Path
import torch
report = {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
          'cuda_runtime': torch.version.cuda, 'cuda_available': torch.cuda.is_available()}
if torch.cuda.is_available():
    report['gpu'] = torch.cuda.get_device_name(0)
    report['gpu_vram_gib'] = torch.cuda.get_device_properties(0).total_memory / 2**30
print(json.dumps(report, indent=2))
assert torch.cuda.is_available(), 'No GPU is active. Select a GPU accelerator or check account access/quota.'


In [ ]:
# Small hardware check only: this is not DTU-Net and does not train on images.
torch.manual_seed(230224)
device = torch.device('cuda:0')
model = torch.nn.Conv2d(3, 3, kernel_size=3, padding=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(2, 3, 64, 64, device=device)
target = torch.randn_like(x)
losses = []
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), target)
    assert torch.isfinite(loss)
    loss.backward()
    assert all(torch.isfinite(p.grad).all() for p in model.parameters())
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
torch.cuda.synchronize()
report['small_optimization_check'] = 'passed'
report['check_losses'] = losses
out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
path = out / 'gpu_environment.json'
path.write_text(json.dumps(report, indent=2))
print('GPU check passed. Saved:', path)
print('GPU:', report['gpu'])


## Next step
Keep the printed GPU name and the JSON report. The paper-model notebook will need a separately tested dependency set, data, and checkpoint/resume support. Passing this check is not proof that the paper configuration fits in GPU memory. Do not install the original requirements list blindly.